In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys
import numpy as np
import keras
project_path = "/content/drive/My Drive/Flight-Safety"
os.chdir(project_path)
sys.path.append(project_path)

In [ ]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!pip install import-ipynb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.5 MB/s eta 0:00:00


In [ ]:
import import_ipynb
from load_process_data import load_data, preprocess_data, truncate_pad_data, shuffle_data, truncate_data
from models import BaseGRU, MultiHeadCnnRnn, MultiHeadAttention, GatedTransformerNetwork
from keras.regularizers import l2
from model_eval import cross_validate
from visualize import visualize_accuracies


In [ ]:
# Define hyperparameters

MAX_FEATURES = 81
input_shape = (81, 23)  # 81 timesteps with 23 features
num_folds = 5  # Cross-validation
epochs = 30
percentage_map = {
    (100, 100): 1  # all data with 75-100% of the original length
}

test_splits = {(100, 100): 1}

# GRU hyperparameters

learning_rate = 0.002
dropout = 0.1
recurrent_dropout = 0.1
kernel_regularizer = l2(0.01)  # Regularization strength for kernel
recurrent_regularizer = l2(0.01)  # Regularization strength for recurrent connections

# CNN hyperparameters

kernel_sizes = [8, 5, 3]
filters = [16, 32, 64]
learning_rate = 0.001
weight_decay = 0.01

# Transformer hyperparameters

t_filters = 32
t_kernel_size = 3
embedding_dim = 64
t_layers = 1
t_heads = 4
t_dropout = 0.2
t_learning_rate = 1e-4
t_weight_decay = 1e-4

In [ ]:
# Create models with regularization

GRUModel = BaseGRU(input_shape=input_shape,
                learning_rate=learning_rate,
                dropout=dropout,
                recurrent_dropout=recurrent_dropout,
                kernel_regularizer=kernel_regularizer,
                recurrent_regularizer=recurrent_regularizer)

MultiHeadModel = MultiHeadCnnRnn(input_shape=input_shape,
                                 kernel_sizes=kernel_sizes,
                                 filters=filters,
                                 learning_rate=learning_rate,
                                 weight_decay=weight_decay,
                                 dropout=dropout,
                                 recurrent_dropout=recurrent_dropout,
                                 kernel_regularizer=kernel_regularizer,
                                 recurrent_regularizer=recurrent_regularizer)

AttentionModel = MultiHeadAttention(input_shape=input_shape,
                                    kernel_sizes=kernel_sizes,
                                    filters=filters,
                                    learning_rate=learning_rate,
                                    weight_decay=weight_decay,
                                    dropout=dropout,
                                    recurrent_dropout=recurrent_dropout,
                                    kernel_regularizer=kernel_regularizer,
                                    recurrent_regularizer=recurrent_regularizer,
                                    attention_block_type='block1')

GTNModel = GatedTransformerNetwork(input_shape=input_shape,
                                   conv_filters=t_filters,
                                   conv_kernel_size=t_kernel_size,
                                   embedding_dim=embedding_dim,
                                   transformer_layers=t_layers,
                                   num_heads=t_heads,
                                   dropout=t_dropout,
                                   learning_rate=t_learning_rate,
                                   weight_decay=t_weight_decay)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Feature amount: 23


In [ ]:
data_path = "/content/drive/MyDrive/Flight-Safety/data/"

train_y_list, test_y_list, train_X_list, test_X_list = load_data(data_path)

# Preprocessing data (reshape + standardization)

train_x_list_filtered, test_x_list_filtered = preprocess_data(train_X_list, test_X_list)
print(train_x_list_filtered[0].shape)
print(train_y_list[0].shape)
print(test_x_list_filtered[0].shape)
print(test_y_list[0].shape)

Loading data ...
Data shapes: 
train_x_list: (5,)
training data within each fold: (16063, 81, 25)
train_y_list: (5,)
training label within each fold: (16063,)
Preprocessing data ...
(16063, 81, 23)
(16063,)
(4016, 81, 23)
(4016,)


In [ ]:
# Reshaping time series train / test data shape for real-time forecasting

train_x_list_random = []
train_y_list_random = []
test_x_list_random = []
test_y_list_random = []

for i, fold in enumerate(train_x_list_filtered):
    train_x_list_truncated = truncate_pad_data(fold, percentage_map, MAX_FEATURES)
    train_x_list_shuffled, train_y_list_shuffled = shuffle_data(train_x_list_truncated, train_y_list[i])
    train_x_list_random.append(train_x_list_shuffled)
    train_y_list_random.append(train_y_list_shuffled)

for i, fold in enumerate(test_x_list_filtered):
    test_x_list_truncated = truncate_pad_data(fold, percentage_map, MAX_FEATURES)
    test_x_list_shuffled, test_y_list_shuffled = shuffle_data(test_x_list_truncated, test_y_list[i])
    test_x_list_random.append(test_x_list_shuffled)
    test_y_list_random.append(test_y_list_shuffled)


In [ ]:
print(f"Fold shape: {train_x_list_random[0].shape}")
print(f"Fold shape: {train_y_list_random[0].shape}")
print(f"Fold shape: {test_x_list_random[0].shape}")
print(f"Fold shape: {test_y_list_random[0].shape}")

Fold shape: (16063, 81, 23)
Fold shape: (16063,)
Fold shape: (4016, 81, 23)
Fold shape: (4016,)


In [ ]:
import time
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

def cross(model, num_folds, train_x_list, train_y_list, test_x_list, test_y_list, epochs=10):
    train_accuracies = []
    test_accuracies = []
    train_losses = []
    test_losses = []
    train_f1_scores = []
    test_f1_scores = []
    train_auc_scores = []
    test_auc_scores = []
    train_precision_scores = []
    test_precision_scores = []
    train_recall_scores = []
    test_recall_scores = []
    training_times = []


    all_train_accuracies = []
    all_test_accuracies = []

    for i in range(num_folds):
        print(f"Training on fold {i+1}/{num_folds}")

        # Extract the training and test data for this fold
        train_X = train_x_list[i]
        train_y = train_y_list[i]
        test_X = test_x_list[i]
        test_y = test_y_list[i]
        keras.backend.clear_session()

        # Re-initialize the model for each fold
        if isinstance(model, BaseGRU):
            new_model = BaseGRU(input_shape=input_shape,
                learning_rate=learning_rate,
                dropout=dropout,
                recurrent_dropout=recurrent_dropout,
                kernel_regularizer=kernel_regularizer,
                recurrent_regularizer=recurrent_regularizer)
        elif isinstance(model, MultiHeadCnnRnn):
             new_model = MultiHeadCnnRnn(input_shape=input_shape,
                                 kernel_sizes=kernel_sizes,
                                 filters=filters,
                                 learning_rate=learning_rate,
                                 weight_decay=weight_decay,
                                 dropout=dropout,
                                 recurrent_dropout=recurrent_dropout,
                                 kernel_regularizer=kernel_regularizer,
                                 recurrent_regularizer=recurrent_regularizer)
        elif isinstance(model, MultiHeadAttention):
             new_model = MultiHeadAttention(input_shape=input_shape,
                                    kernel_sizes=kernel_sizes,
                                    filters=filters,
                                    learning_rate=learning_rate,
                                    weight_decay=weight_decay,
                                    dropout=dropout,
                                    recurrent_dropout=recurrent_dropout,
                                    kernel_regularizer=kernel_regularizer,
                                    recurrent_regularizer=recurrent_regularizer,
                                    attention_block_type='block1')
        elif isinstance(model, GatedTransformerNetwork):
            new_model = GatedTransformerNetwork(input_shape=input_shape,
                                   conv_filters=t_filters,
                                   conv_kernel_size=t_kernel_size,
                                   embedding_dim=embedding_dim,
                                   transformer_layers=t_layers,
                                   num_heads=t_heads,
                                   dropout=t_dropout,
                                   learning_rate=t_learning_rate,
                                   weight_decay=t_weight_decay)
        else:
            raise ValueError("Unsupported model type")


        # Fit the model
        start_time = time.time()
        history = new_model.fit(train_X, train_y, epochs=epochs, validation_data=(test_X, test_y))
        end_time = time.time()
        training_times.append(end_time - start_time)


        # Evaluate predictions
        train_pred_prob = new_model.predict(train_X)
        test_pred_prob = new_model.predict(test_X)

        # Convert probabilities to binary predictions for classification metrics
        train_pred = (train_pred_prob > 0.5).astype(int)
        test_pred = (test_pred_prob > 0.5).astype(int)


        # Calculate metrics for the last epoch
        train_accuracies.append(history.history['accuracy'][-1])
        test_accuracies.append(history.history['val_accuracy'][-1])
        train_losses.append(history.history['loss'][-1])
        test_losses.append(history.history['val_loss'][-1])
        train_f1_scores.append(f1_score(train_y, train_pred))
        test_f1_scores.append(f1_score(test_y, test_pred))
        train_auc_scores.append(roc_auc_score(train_y, train_pred_prob))
        test_auc_scores.append(roc_auc_score(test_y, test_pred_prob))
        train_precision_scores.append(precision_score(train_y, train_pred))
        test_precision_scores.append(precision_score(test_y, test_pred))
        train_recall_scores.append(recall_score(train_y, train_pred))
        test_recall_scores.append(recall_score(test_y, test_pred))

        # Storing each epoch's training and testing accuracies
        all_train_accuracies.append(history.history['accuracy'])
        all_test_accuracies.append(history.history['val_accuracy'])

    # Calculate the average accuracies and losses across all folds
    average_train_accuracy = np.mean(train_accuracies)
    average_test_accuracy = np.mean(test_accuracies)
    average_train_loss = np.mean(train_losses)
    average_test_loss = np.mean(test_losses)
    average_train_f1 = np.mean(train_f1_scores)
    average_test_f1 = np.mean(test_f1_scores)
    average_train_auc = np.mean(train_auc_scores)
    average_test_auc = np.mean(test_auc_scores)
    average_train_precision = np.mean(train_precision_scores)
    average_test_precision = np.mean(test_precision_scores)
    average_train_recall = np.mean(train_recall_scores)
    average_test_recall = np.mean(test_recall_scores)
    average_training_time = np.mean(training_times)


    print(f"Average training accuracy: {average_train_accuracy:.4f}")
    print(f"Average testing accuracy: {average_test_accuracy:.4f}")
    print(f"Average training loss: {average_train_loss:.4f}")
    print(f"Average testing loss: {average_test_loss:.4f}")
    print(f"Average training F1-score: {average_train_f1:.4f}")
    print(f"Average testing F1-score: {average_test_f1:.4f}")
    print(f"Average training AUC: {average_train_auc:.4f}")
    print(f"Average testing AUC: {average_test_auc:.4f}")
    print(f"Average training Precision: {average_train_precision:.4f}")
    print(f"Average testing Precision: {average_test_precision:.4f}")
    print(f"Average training Recall: {average_train_recall:.4f}")
    print(f"Average testing Recall: {average_test_recall:.4f}")
    print(f"Average training time (seconds): {average_training_time:.4f}")


    return (all_train_accuracies, all_test_accuracies,
            train_f1_scores, test_f1_scores,
            train_auc_scores, test_auc_scores,
            train_precision_scores, test_precision_scores,
            train_recall_scores, test_recall_scores,
            training_times)

In [ ]:
train_accuracies, test_accuracies = cross(GRUModel, num_folds, train_x_list_random,
                                                   train_y_list_random, test_x_list_random, test_y_list_random, epochs)

Training on fold 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
502/502 ━━━━━━━━━━━━━━━━━━━━ 133s 240ms/step - accuracy: 0.6266 - loss: 1.0292 - val_accuracy: 0.7495 - val_loss: 0.5999
Epoch 2/30
502/502 ━━━━━━━━━━━━━━━━━━━━ 136s 237ms/step - accuracy: 0.7512 - loss: 0.6016 - val_accuracy: 0.8528 - val_loss: 0.4841
Epoch 3/30
502/502 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step - accuracy: 0.7799 - loss: 0.5654